# Temporal LoRA Adapters

<a href="https://colab.research.google.com/github/Text-Machine/temporal-adapters/blob/train/temporal-adapters.ipynb" target="_parent\"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/> </a>

This notebook trains temporal LoRA adapters on ECCO data.

Adapters are trained per decade (window), and each window moves in 5-year steps.

You can either train a new adapter from scratch or continue training from an existing saved LoRA adapter by setting `resume_adapter_path`.

In [ ]:
!pip -qqq install trl peft

In [ ]:
# Import necessary libraries
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from datasets import load_dataset
import json
import logging
import re
from datasets import Dataset
import pandas as pd
from pathlib import Path
from typing import Optional, Dict, Any
from peft import LoraConfig, PeftModel
from trl import SFTTrainer, SFTConfig



In [2]:
# Download NLTK punkt tokenizer for sentence splitting
import nltk
nltk.download('punkt_tab')
nltk.download('punkt', quiet=True)  
from nltk.tokenize import sent_tokenize

print("✓ NLTK punkt downloaded")

✓ NLTK punkt downloaded


[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/kasparbeelen/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [3]:
if torch.cuda.is_available():
    device = "cuda"
    print(f"Using CUDA GPU: {torch.cuda.get_device_name()}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f}GB")
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    device = "mps"
    print("Using Apple MPS")
else:
    device = "cpu"
    print("Using CPU - you will need to use a GPU to train models")



Using Apple MPS


In [4]:
# Authenticate with Hugging Face (optional, for private models)
from huggingface_hub import login
login()  # Uncomment if you need to access private models

In [7]:
# Uncomment the line below to download the data from Google Drive using gdown
#!gdown 11wfdV7j1TBv_i9XOiT8G8V4NxnJTxezz

In [8]:
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

print("✓ Libraries imported successfully")

✓ Libraries imported successfully


In [21]:
def load_csv_as_dataset(csv_paths):
    """Load CSV files and convert to Hugging Face Dataset."""
    all_data = []
    
    for csv_path in csv_paths:
        logger.info(f"Loading {Path(csv_path).name}...")
        df = pd.read_csv(csv_path)
        logger.info(f"  Rows: {len(df)}, Columns: {list(df.columns)}")
        all_data.append(df)
    
    combined_df = pd.concat(all_data, ignore_index=True)
    logger.info(f"Total rows: {len(combined_df)}")
    
    dataset = Dataset.from_pandas(combined_df)
    return dataset

def text_preprocess_function(batch, min_chars: int = 40, min_words: int = 6):
    """Convert page-level examples to sentence-level training rows."""
    texts = []
    # years = []

    page_texts = batch.get("page_text", [])
    # converted_dates = batch.get("converted_date", [None] * len(page_texts))

    for page_text in page_texts: # zip(page_texts, converted_dates)
        if page_text is None:
            continue

        page_text = re.sub(r"\s+", " ", str(page_text)).strip()
        if not page_text:
            continue

        for sent in sent_tokenize(page_text):
            #sent = re.sub(r"\s+", " ", sent).strip()
            if len(sent) < min_chars or len(sent.split()) < min_words:
                continue
            texts.append(sent)
            #years.append(year)

    return {"text": texts} # , "converted_date": years

In [10]:
base_path = '../data-processing-code/data'
csv_path = Path(base_path).glob('*_cleaned.csv')
dataset = load_csv_as_dataset(csv_path)

INFO:__main__:Loading ecco_pages_cleaned.csv...
INFO:__main__:  Rows: 169051, Columns: ['author', 'place', 'date', 'page_text', 'converted_date']
INFO:__main__:Total rows: 169051


In [ ]:
adapters_windows = [(year, year+10) for year in list(range(1700,1795,5))]


[(1700, 1710), (1705, 1715)]

In [ ]:
# Choose one decade for LoRA fine-tuning
selected_window = adapters_windows[0]  # e.g., (1700, 1710)
start_year, end_year = selected_window

# Optional: continue training from an existing saved LoRA adapter
# Example: resume_adapter_path = "lora-adapter_1700_1710/checkpoint-2426"
resume_adapter_path: Optional[str] = None

# Optional: if you also want to resume optimizer/scheduler/trainer state
# set this to a Trainer checkpoint directory (same format as above).
resume_trainer_checkpoint: Optional[str] = None

In [ ]:
base_model_name = "meta-llama/Meta-Llama-3-8B"
model = AutoModelForCausalLM.from_pretrained(base_model_name)
tokenizer = AutoTokenizer.from_pretrained(base_model_name)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

if resume_adapter_path:
    model = PeftModel.from_pretrained(
        model,
        resume_adapter_path,
        is_trainable=True,
    )
    print(f"Loaded existing LoRA adapter for continued training: {resume_adapter_path}")
else:
    print("Starting a new LoRA adapter from scratch for this window.")

In [ ]:
dataset_decade = dataset.filter(
    lambda x: start_year <= x["converted_date"] < end_year
    )

print(f"Selected decade: {start_year}-{end_year}")
print(f"Page-level rows in selected decade: {len(dataset_decade):,}")

dataset_decade_processed = dataset_decade.map(
    text_preprocess_function,
    batched=True,
    remove_columns=dataset_decade.column_names,
    desc="Converting pages to sentence-level dataset",
)

print(f"Sentence-level rows: {len(dataset_decade_processed):,}")


Filter:   0%|          | 0/169051 [00:00<?, ? examples/s]

Selected decade: 1700-1710
Page-level rows in selected decade: 7,764


In [ ]:
# 1) Configure LoRA (used only when training a new adapter)
peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

# 2) Configure trainer
sft_config = SFTConfig(
    output_dir=f"lora-adapter_{start_year}_{end_year}",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    packing=True,
    dataset_text_field="text",
)

trainer_kwargs = dict(
    model=model,
    args=sft_config,
    train_dataset=dataset_decade_processed,
    processing_class=tokenizer,
 )

if not resume_adapter_path:
    trainer_kwargs["peft_config"] = peft_config

trainer = SFTTrainer(**trainer_kwargs)

trainer.train(resume_from_checkpoint=resume_trainer_checkpoint)